# TowerIQ Silver Layer Inspection

Use this notebook to inspect valid-record Silver tables and enriched Silver event tables.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.bronze_ingestion import build_storage_path
from src.ingestion.schemas import RAW_SCHEMAS
from src.jobs.run_silver_transformations import SOURCE_TABLE_BY_ENRICHED_TABLE
from src.utils.config import load_config
from src.utils.spark import create_spark_session

config = load_config("configs/local.yaml")
spark_config = config["spark"]
paths = config["paths"]

spark = create_spark_session(
    app_name="TowerIQ-SilverInspection",
    master=spark_config["master"],
    aqe_enabled=bool(spark_config["adaptive_query_execution"]),
    use_pyspark_package=bool(spark_config.get("use_pyspark_package", True)),
)
spark

## Load Silver Tables

In [ ]:
silver_tables = {
    table_name: spark.read.parquet(build_storage_path(paths["silver"], "tiny", table_name))
    for table_name in RAW_SCHEMAS
}
enriched_tables = {
    table_name: spark.read.parquet(build_storage_path(paths["silver"], "tiny", "enriched", table_name))
    for table_name in SOURCE_TABLE_BY_ENRICHED_TABLE
}

for table_name, df in silver_tables.items():
    df.createOrReplaceTempView(f"silver_{table_name}")
    print("silver", table_name, df.count())

for table_name, df in enriched_tables.items():
    df.createOrReplaceTempView(table_name)
    print("enriched", table_name, df.count())

## Q1. What valid-record Silver row counts exist?

In [ ]:
for table_name, df in silver_tables.items():
    print(f"{table_name}: {df.count():,}")

## Q2. What enriched Silver tables exist?

In [ ]:
for table_name, df in enriched_tables.items():
    print(f"{table_name}: {df.count():,} rows, {len(df.columns)} columns")

## Q3. Inspect enriched network events

In [ ]:
spark.sql("""
SELECT
  event_id,
  event_date,
  event_hour,
  tower_id,
  region_name,
  tower_type,
  subscriber_id,
  customer_segment,
  plan_type,
  manufacturer,
  network_type,
  status
FROM network_events_enriched
LIMIT 10
""").show(truncate=False)

## Q4. How are calls enriched by region and plan type?

In [ ]:
spark.sql("""
SELECT
  region_name,
  plan_type,
  COUNT(*) AS calls,
  SUM(CASE WHEN is_dropped_call THEN 1 ELSE 0 END) AS dropped_calls,
  SUM(CASE WHEN is_failed_call THEN 1 ELSE 0 END) AS failed_calls
FROM calls_enriched
GROUP BY region_name, plan_type
ORDER BY calls DESC
LIMIT 15
""").show(truncate=False)

## Q5. Which enriched data sessions have the highest data usage?

In [ ]:
spark.sql("""
SELECT
  session_id,
  event_date,
  tower_id,
  region_name,
  network_type,
  customer_segment,
  plan_type,
  total_mb,
  latency_ms,
  session_status
FROM data_sessions_enriched
ORDER BY total_mb DESC
LIMIT 10
""").show(truncate=False)

## Q6. Which tower alarms are critical?

In [ ]:
spark.sql("""
SELECT
  alarm_id,
  event_date,
  tower_id,
  region_name,
  tower_type,
  alarm_type,
  severity,
  alarm_status,
  is_critical_alarm
FROM tower_alarms_enriched
WHERE is_critical_alarm = true
LIMIT 20
""").show(truncate=False)

## Stop Spark

In [ ]:
spark.stop()